# Droplet rebound simulation - Initial conditions (free fall droplet static)

This notebook is used for computing initial conditions for the velocity field of the droplet rebound test case. Since the droplet has an impact velocity (VelocityZ#A), we increase the corresponding field successivley form 0 to the desired impact velocity value.
The resulting solution will be stored in designated database, from which the actual simulation, see `DropletRebound.ipynb`, will be restarted.

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 204
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 97
   at Submission#18.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;

In [ ]:
BoSSSshell.WorkflowMgm.Init("DropletReboundInit");

In [ ]:
//ExecutionQueues

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,\\hpccluster\hpccluster-scratch\smuda\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,DC2,<null>,Normal,True,[ \\hpccluster\hpccluster-scratch\smuda\ ]


## Grid creation

The used grid is a cartesain box around the droplet injection location, which is located at `radiusOP` (in $r$-direction) away from the center of the rotating disk.

In [ ]:
static public Grid3D RotatingDiskSector_CartesianCutOut(double radiusOP, double l_radial, double l_azimuthal, double h_axial, int res_radial, int res_azimuthal, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(radiusOP - (l_radial / 2.0), radiusOP + (l_radial / 2.0), res_radial + 1);    // radial direction
    double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 2.0, l_azimuthal / 2.0, res_azimuthal + 1);    // azimuthal direction
    // double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 3.0, (2.0 * l_azimuthal) / 3.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_CartesianCutOut_{res_radial}x{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "velocity_inlet_outerFlow");
    grd.EdgeTagNames.Add(2, "pressure_outlet_top");
    grd.EdgeTagNames.Add(3, "pressure_outlet_back");
    grd.EdgeTagNames.Add(4, "pressure_outlet_front");
    grd.EdgeTagNames.Add(5, "pressure_outlet_upstream");
    grd.EdgeTagNames.Add(6, "pressure_outlet_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.z.Abs() <= 1e-8)
            et = 1;
        if ((X.z - h_axial).Abs() <= 1e-8)
            et = 2;
        if (((X.x - radiusOP) + (l_radial / 2.0)).Abs() <= 1e-8)
            et = 3;
        if (((X.x - radiusOP) - (l_radial / 2.0)).Abs() <= 1e-8)
            et = 4;
        if ((X.y + (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 5;
        if ((X.y - (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 6;
        // if ((X.y + (l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 5;
        // if ((X.y - (2.0 * l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 6;

        return et;
    });    

    return grd;
}

## simulation setup

In [ ]:
double radiusOP = 78e-3; // operating point (droplet injection point) -> Re = radiusOp / Lstar
double density = 1.19;  
double kinViscosity = 1.52e-5; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double omega = 1820.5128205128206; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(kinViscosity / omega);  // used for non-dimensionalization of the flow fields
Lstar

9.137448098155134E-05

Reynolds number at the point of injection

In [ ]:
double reynolds = radiusOP / Lstar;
reynolds

853.6300196960717

Boundary layer (BL) thickness

In [ ]:
double zBL = 5.5;   // dimesionless value
double zBLstar = zBL * Lstar;
zBLstar

0.0005025596453985323

Height of the computational domain (roughly 3 times the BL thickness)

In [ ]:
double zTop = 15;
double zTopStar = zTop * Lstar;
zTopStar

0.00137061721472327

In [ ]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { " +
    "double radiusOP = 78e-3;" +
    "double radiusDrop = 0.205e-3;" +
    "double initHeight = 75e-5;" +
    "return Math.Sqrt((X[0] - radiusOP).Pow2() + X[1].Pow2() + (X[2] - initHeight).Pow2()) - radiusDrop; } "
);

In [ ]:
Formula ImpactVelocity = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return 0.112; } "
);

## Control settings

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
var C = new XNSE_Control();

int k = 3;
C.SetDGdegree(k);


// physical parameters
double densityDrop = 998.37;
C.PhysicalParameters.rho_A = densityDrop;
C.PhysicalParameters.mu_A = densityDrop * 0.000001028;

C.PhysicalParameters.rho_B = density;
C.PhysicalParameters.mu_B = density * kinViscosity;

C.PhysicalParameters.Sigma = 72.9e-3;

C.PhysicalParameters.IncludeConvection = true;

    
// set grid
double l_radial = 1.0 * zTopStar; 
double l_azimuthal = 1.0 * zTopStar;
// double l_azimuthal = (3.0/2.0) * zTopStar;
double h_axial = 1 * zTopStar; 

int res_global = 8;
int res_radial = (int)(1.0 * res_global); 
int res_azimuthal = (int)(1.0 * res_global);
// int res_azimuthal = (3 * res_global) / 2;
int res_axial = 1 * res_global;

Grid3D grd = RotatingDiskSector_CartesianCutOut(radiusOP, l_radial, l_azimuthal, h_axial, res_radial, res_azimuthal, res_axial);
C.SetGrid(grd);

// boundary conditions
//C.AddBoundaryValue("velocity_inlet_outerFlow", "VelocityZ#A", ImpactVelocity);
C.AddBoundaryValue("velocity_inlet_outerFlow", "VelocityZ#B", ImpactVelocity);


// initial conditions
//C.AddInitialValue("VelocityZ#B", ImpactVelocity);

C.AddInitialValue("Phi", PhiFunc);


C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;


// C.SkipSolveAndEvaluateResidual = true;

C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 200;

// C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.Timestepper_LevelSetHandling = LevelSetHandling.None;
C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.dtFixed = 1.0e-1;
C.NoOfTimesteps = 1000;
    
{
    C.AdaptiveMeshRefinement = true;
    int AMRlevel = 1;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel });
    C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel });
    C.AMR_startUpSweeps = AMRlevel + 1;
}

// C.PostprocessingModules.Add(new EnergyLogging());
    
C.SessionName = "DropletRebound_8x8x8AMR1_freeFall10vH_InitState_transient4";
    
Controls.Add(C);

## Launch job

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,DropletRebound_8x8x8AMR1_freeFall10vH_InitState_transient4


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    oneJob.NumberOfMPIProcs = 1;
    oneJob.Activate(myBatch); 
}

## Move restart session

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletReboundInit");
var initDB = databases.Pick(1);
initDB

{ Session Count = 13; Grid Count = 39; Path = \\hpccluster\hpccluster-scratch\smuda\DropletReboundInit }

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletRebound");
var runDB = databases.Pick(2);
runDB

Opening existing database '\\hpccluster\hpccluster-scratch\smuda\DropletRebound'.


{ Session Count = 18; Grid Count = 36; Path = \\hpccluster\hpccluster-scratch\smuda\DropletRebound }

In [ ]:
initDB.Sessions.Pick(1).Move(runDB);

## Postprocessing

In [ ]:
OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletReboundInit");

In [ ]:
databases.Pick(1).Sessions

#0: DropletReboundInit	DropletRebound_8x8x8AMR1_freeFall10vH_InitState_transient*	11/02/2023 17:18:42	e9f2fc5e...
#1: DropletReboundInit	DropletRebound_12x12x16AMR1_freeFall10vH_InitState_pOut*	10/30/2023 14:15:27	bd4f19cd...
#2: DropletReboundInit	DropletRebound_8x8x16AMR1_freeFall10vH_InitState2_pOut*	10/30/2023 14:13:15	44e23c5f...
#3: DropletReboundInit	DropletRebound_8x8x16AMR1_freeFall25vH_InitState_pOut*	10/28/2023 21:01:56	a6ecdf50...
#4: DropletReboundInit	DropletRebound_8x8x16AMR1_freeFall10vH_InitState_pOut*	10/28/2023 20:59:13	6e6ae1b5...
#5: DropletReboundInit	DropletRebound_8x8x16AMR1_freeFall_InitState_pOut*	10/28/2023 15:01:30	b05b61cd...
#6: DropletReboundInit	DropletRebound_8x8x16AMR1_freeFall_InitState*	10/27/2023 17:01:25	37703b44...
#7: DropletReboundInit	DropletRebound_8x8x16AMR2_topFrontPrescribed_InitState*	10/20/2023 13:56:57	f21a986c...
#8: DropletReboundInit	DropletRebound_8x8x16AMR2_Picard5_InitState	10/16/2023 14:41:12	4d2a54d7...
#9: DropletReboundInit	Dro

In [ ]:
var sess = databases.Pick(1).Sessions.Pick(0);
sess

DropletReboundInit	DropletRebound_8x8x8AMR1_freeFall10vH_InitState_transient*	11/02/2023 17:18:42	e9f2fc5e...

In [ ]:
sess.Timesteps.Skip(3).Every(10)

#0:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#1:  { Time-step: 10; Physical time: 1E-05s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#2:  { Time-step: 20; Physical time: 2.0000000000000005E-05s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#3:  { Time-step: 30; Physical time: 3.000000000000001E-05s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-

In [ ]:
sess.Timesteps.Skip(3).Every(10).Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletReboundInit__DropletRebound_8x8x8AMR1_freeFall10vH_InitState_transient__e9f2fc5e-6209-4c61-b469-b7f440785cc7


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletReboundInit__DropletRebound_8x8x8AMR1_freeFall10vH_InitState_transient__e9f2fc5e-6209-4c61-b469-b7f440785cc7

### Velocity profiles

In [ ]:
int tIdx = 2;
sess.Timesteps.Pick(tIdx)

 { Time-step: 0.2; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }

In [ ]:
DGField[] solutionFields = new DGField[4];
solutionFields[0] = sess.Timesteps.Pick(tIdx).GetField("VelocityX");
solutionFields[1] = sess.Timesteps.Pick(tIdx).GetField("VelocityY");
solutionFields[2] = sess.Timesteps.Pick(tIdx).GetField("VelocityZ");
solutionFields[3] = sess.Timesteps.Pick(tIdx).GetField("Pressure");

In [ ]:
double[][] solutionValues = new double[4][];
double[] evalPoint = new double[] {radiusOP - 1.0e-5, 1.0e-5};

int numVal = 100;
double stepSize = zTopStar / numVal; 
double[] zStarValues = Enumerable.Range(0, numVal+1).Select(idx => idx * stepSize).ToArray();

// velocity fields
for (int j = 0; j < 3; j++) {
    solutionValues[j] = new double[numVal+1];
    for (int i = 0; i <= numVal; i++) {
        try {
            solutionValues[j][i] = solutionFields[j].ProbeAt(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]});
        }
        catch {
            Console.WriteLine("evalPoint = ( {0}, {1}, {2} ) not found", evalPoint[0], evalPoint[1], zStarValues[i]);
        }
    }
}
// pressure field (correct to same pressure level at wall p=0)
solutionValues[3] = new double[numVal+1];
double pressure0 = solutionFields[3].ProbeAt(new double[] {evalPoint[0], evalPoint[1], 0.0});
for (int i = 0; i <= numVal; i++) {
    try {
        solutionValues[3][i] = solutionFields[3].ProbeAt(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}) - pressure0;
    }
    catch {
        Console.WriteLine("evalPoint = ( {0}, {1}, {2} ) not found", evalPoint[0], evalPoint[1], zStarValues[i]);
    }
}

In [ ]:
double[][] varValues = new double[4][];
varValues[0] = new double[numVal+1];
varValues[1] = new double[numVal+1];
varValues[2] = new double[numVal+1];
varValues[3] = new double[numVal+1];

for (int i = 0; i <= numVal; i++) {
    varValues[0][i] = vonKarmanHAM_velX.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    varValues[1][i] = vonKarmanHAM_velY.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0); 
    varValues[2][i] = vonKarmanHAM_velZ.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
    varValues[3][i] = vonKarmanHAM_P.Evaluate(new double[] {evalPoint[0], evalPoint[1], zStarValues[i]}, 0.0);
}

In [ ]:
var fmt1 = new PlotFormat();    // blue - BoSSS
fmt1.LineColor = LineColors.Blue;
var fmt2 = new PlotFormat();    // red - Analytic

var gp = new Gnuplot();
gp.SetMultiplot(2,2);

int idx = 0;
for (int i = 0; i < 2; i++)
for (int j = 0; j < 2; j++) {
    Plot2Ddata plt = new Plot2Ddata(); 
    plt.AddDataGroup("BoSSS", solutionValues[idx], zStarValues, fmt1);
    plt.AddDataGroup("Analytic", varValues[idx], zStarValues, fmt2);
    gp.SetSubPlot(i,j, varNames[idx]);
    plt.ToGnuplot(gp);
    idx++;
}
//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:1000)

Error: (14,24): error CS0103: The name 'varNames' does not exist in the current context